In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load the dataset
Q1_path = os.path.join(path, 'Q1_data.csv')
df_Q1 = pd.read_csv(Q1_path)
print(f"Dataset shape: {df_Q1.shape}")
df_Q1.head()

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:
print("\n=== First few rows ===")
print(df_Q1.head())

In [ ]:
# Task 3: Write your code here:
print("\n=== Dataset Information ===")
df_Q1.info()

In [ ]:
# Task 4: Write your code here:
print("\n=== Statistical Description ===")
print(df_Q1.describe())

In [ ]:
# Task 5: Write your code here:
# delivery time distribution (target variable)
plt.figure(figsize=(10, 6))
plt.hist(df_Q1['Delivery_Time'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.title('Distribution of Delivery Time')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:
df_Q1 = df_Q1.drop('Order_ID', axis=1)
print(f"Shape after dropping Order_ID: {df_Q1.shape}")

In [ ]:
# Task 2: Write your code here:
print("\n=== Missing Values ===")
print(df_Q1.isnull().sum())

In [ ]:
# Fill numerical missing values with median
for col in df_Q1.select_dtypes(include=[np.number]).columns:
    if df_Q1[col].isnull().sum() > 0:
        df_Q1[col].fillna(df_Q1[col].median(), inplace=True)

# Fill categorical missing values with mode
for col in df_Q1.select_dtypes(include=['object']).columns:
    if df_Q1[col].isnull().sum() > 0:
        df_Q1[col].fillna(df_Q1[col].mode()[0], inplace=True)

print("\nMissing values after handling:")
print(df_Q1.isnull().sum())

In [ ]:
# Task 3: Write your code here:
print(f"\n=== Duplicates ===")
print(f"Number of duplicates: {df_Q1.duplicated().sum()}")
df_Q1 = df_Q1.drop_duplicates()
print(f"Shape after removing duplicates: {df_Q1.shape}")

In [ ]:
# Task 4: Write your code here:
print("\n=== Encoding Categorical Variables ===")
categorical_cols = df_Q1.select_dtypes(include=['object']).columns.tolist()
# Remove target if it's in categorical columns
if 'Delivery_Time' in categorical_cols:
    categorical_cols.remove('Delivery_Time')
    print(f"Categorical columns: {categorical_cols}")
# Apply One Hot Encoding
df_Q1 = pd.get_dummies(df_Q1, columns=categorical_cols, drop_first=True)
print(f"Shape after encoding: {df_Q1.shape}")



In [ ]:
# Task 5: Write your code here:
# Separate features and target
X = df_Q1.drop('Delivery_Time', axis=1)
y = df_Q1['Delivery_Time']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n=== Feature Scaling Applied ===")
print(f"Features shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Task 6: Write your code here:
# Since this is a regression problem (continuous target), we check the distribution
print("\n=== Target Distribution Check ===")
print(f"Target mean: {y.mean():.2f}")
print(f"Target median: {y.median():.2f}")
print(f"Target std: {y.std():.2f}")
print(f"Target skewness: {y.skew():.2f}")

In [ ]:
# Task 1: Write your code here:
# Task 1: Split features and target (already done above)
print("\n=== Model Training ===")

In [ ]:
# Task 2,3,4,5: Write your code here:
# For regression problems, we use KFold (not StratifiedKFold which is for classification)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train RandomForest model
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    # Calculate MAE
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)
    models.append(model)

    print(f"Fold {fold} - MAE: {mae:.4f}")

# Print averaged score
print(f"\n=== Cross-Validation Results ===")
print(f"Average MAE across all folds: {np.mean(mae_scores):.4f} (+/- {np.std(mae_scores):.4f})")

# Train final model on full data for feature importance and predictions
final_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
final_model.fit(X_scaled, y)

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X_scaled.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Top 15 Feature Importances from Random Forest Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred_final = final_model.predict(X_scaled)

plt.figure(figsize=(12, 6))
plt.hist(y, bins=30, alpha=0.5, label='Actual Delivery Time', edgecolor='black')
plt.hist(y_pred_final, bins=30, alpha=0.5, label='Predicted Delivery Time', edgecolor='black')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.title('Distribution of Actual vs Predicted Delivery Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#  Ensemble with RandomForest and CatBoost
!pip install catboost
from catboost import CatBoostRegressor

print("\n=== Bonus: Ensemble Model ===")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train RandomForest model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)

    # Train CatBoost model
    cb_model = CatBoostRegressor(iterations=100, random_state=42, verbose=0)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_val)

    # Average predictions
    ensemble_pred = (rf_pred + cb_pred) / 2

    # Calculate MAE on averaged predictions
    mae = mean_absolute_error(y_val, ensemble_pred)
    ensemble_mae_scores.append(mae)

    print(f"Fold {fold} - Ensemble MAE: {mae:.4f}")

# Print averaged score
print(f"\n=== Ensemble Cross-Validation Results ===")
print(f"Average MAE across all folds: {np.mean(ensemble_mae_scores):.4f} (+/- {np.std(ensemble_mae_scores):.4f})")
print(f"\nComparison:")
print(f"Single Model (RandomForest) MAE: {np.mean(mae_scores):.4f}")
print(f"Ensemble Model (RF + CatBoost) MAE: {np.mean(ensemble_mae_scores):.4f}")